# 4 · Graph Queries

*Phasor networks, from the ground up — notebook 4 of 6.*

A whole graph fits in a single hypervector. Give each node a random symbol; encode
an **edge** as the *binding* of its endpoints; encode the **graph** as the *bundle*
of all its edges. To ask "who are node *i*'s neighbours?", **unbind** the node from
the graph memory and **clean up** the result against the node codebook — the
neighbours light up. One vector stores the structure; one unbind queries it.

Ported from the older `phasor_julia` graph demo onto the current API.

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
using PhasorNetworks
using Plots
using Random: Xoshiro
using LinearAlgebra: triu

## Encoding and querying

In [ ]:
function generate_er_graph(n, p, rng)
    adj = rand(rng, Float64, (n, n)) .< p
    adj = triu(adj); adj = (adj .+ adj') .> 0      # undirected
    for i in 1:n; adj[i, i] = 0; end               # no self-loops
    return adj
end

# graph memory = bundle over edges of bind(src, dst)
function graph_to_vector(graph, nodes)
    edges = findall(graph)
    edge_syms = [v_bind(nodes[e[1], :], nodes[e[2], :]) for e in edges]
    edge_mat = stack(edge_syms, dims=1)            # (n_edges, d)
    return v_bundle(edge_mat, dims=1)              # (1, d)
end

# query each node: unbind from memory, clean up against all nodes
function query_edges(emb, nodes)
    n = size(nodes, 1)
    rec = zeros(Float64, n, n)
    for i in 1:n
        q = v_unbind(emb, reshape(nodes[i, :], (1, :)))
        rec[i, :] = vec(similarity_outer(q, nodes, dims=1))
    end
    return rec
end

# rank-based AUROC (no external deps)
function auroc(labels, scores)
    pos = scores[labels]; neg = scores[.!labels]
    (isempty(pos) || isempty(neg)) && return NaN
    c = 0.0
    for a in pos, b in neg
        c += a > b ? 1.0 : (a == b ? 0.5 : 0.0)
    end
    return c / (length(pos) * length(neg))
end

## Store a random graph and read it back

A 20-node Erdős–Rényi graph in 1024-dim space. We reconstruct the full adjacency from the single graph vector.

In [ ]:
rng = Xoshiro(0)
n = 20; d = 1024; p = 0.2

graph = generate_er_graph(n, p, rng)
nodes = random_symbols(rng, (n, d))
emb = graph_to_vector(graph, nodes)
rec = query_edges(emb, nodes)

score = auroc(vec(graph), vec(rec))
println("reconstruction AUROC: ", round(score, digits=3))

In [ ]:
h1 = heatmap(graph, title="true adjacency", yflip=true, aspect_ratio=:equal, colorbar=false)
h2 = heatmap(rec, title="reconstructed (similarity)", yflip=true, aspect_ratio=:equal)
plot(h1, h2, layout=(1, 2), size=(800, 360))

The reconstruction's score distribution separates edges from non-edges — that gap is what the AUROC measures.

In [ ]:
edge_scores = vec(rec)[vec(graph)]
nonedge_scores = vec(rec)[.!vec(graph)]
histogram(nonedge_scores, bins=40, alpha=0.6, label="non-edges", xlabel="similarity", ylabel="count")
histogram!(edge_scores, bins=40, alpha=0.6, label="edges", title="edge vs non-edge similarity")

## Takeaway

Structure becomes a single vector through **bind** (edges) and **bundle** (the
graph), and is queried by **unbind** + **similarity** cleanup. Because every
primitive has an oscillator form (notebook 2), the same store-and-query runs on
oscillating hardware. Next: *learning* with these representations.